In [0]:
%pip install xgboost

In [0]:
import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_percentage_error, r2_score

In [0]:
query = """
SELECT 
    fs.order_date,
    fs.product_fk,
    fs.territory_fk,
    fs.order_quantity,
    fs.unit_price,
    fs.total_due,
    dp.product_name,
    dp.product_category_name,
    dt.country_region_name,
    ds.store_name,
    ds.business_entity_id as store_id
FROM ted_dev.marts.fact_sales fs
JOIN ted_dev.marts.dim_product dp ON fs.product_fk = dp.product_pk
JOIN ted_dev.marts.dim_territory dt ON fs.territory_fk = dt.territory_pk
LEFT JOIN ted_dev.marts.dim_store ds ON fs.sales_person_fk = ds.sales_person_id
ORDER BY fs.order_date
"""

df = spark.sql(query).toPandas()
df['order_date'] = pd.to_datetime(df['order_date'])

print(f"Dados carregados: {len(df):,} registros")
print(f"Período: {df['order_date'].min()} a {df['order_date'].max()}")
print(f"Produtos únicos: {df['product_fk'].nunique()}")
print(f"Lojas distintas: {df['store_id'].nunique()}")

monthly_data = df.groupby([
    'product_fk', 
    'store_id',
    pd.Grouper(key='order_date', freq='M')
]).agg({
    'order_quantity': 'sum',
    'unit_price': 'mean',
    'total_due': 'sum',
    'product_name': 'first',
    'store_name': 'first',
    'country_region_name': 'first'
}).reset_index()

monthly_data = monthly_data.dropna(subset=['store_id'])


# Viabilidade dos modelos de regressão para previsão de demanda

### Objetivo
Avaliar a viabilidade de modelos de regressão para resolver o problema de previsão de demanda, comparando diferentes abordagens e identificando a solução baseada em métricas.

In [0]:
monthly_data_sorted = monthly_data.sort_values(
    by=['product_fk', 'store_id', 'order_date']
).reset_index(drop=True)
modeling_data = monthly_data_sorted.copy()
modeling_data['month'] = modeling_data['order_date'].dt.month
modeling_data['year'] = modeling_data['order_date'].dt.year
modeling_data['quarter'] = modeling_data['order_date'].dt.quarter
lags = [1, 2, 3, 6, 12]
for lag in lags:
    modeling_data[f'quantity_lag{lag}'] = modeling_data.groupby(
        ['product_fk', 'store_id']
    )['order_quantity'].shift(lag)

for window in [3, 6, 12]:
    modeling_data[f'quantity_ma{window}'] = modeling_data.groupby(
        ['product_fk', 'store_id']
    )['quantity_lag1'].rolling(window, min_periods=1).mean().reset_index(level=[0,1], drop=True)

modeling_data['quantity_std6'] = modeling_data.groupby(
    ['product_fk', 'store_id']
)['quantity_lag1'].rolling(6, min_periods=2).std().reset_index(level=[0,1], drop=True)

modeling_data = modeling_data.dropna().reset_index(drop=True)

features = [
    'month', 'year', 'quarter',
    'quantity_lag1', 'quantity_lag2', 'quantity_lag3', 'quantity_lag6', 'quantity_lag12',
    'quantity_ma3', 'quantity_ma6', 'quantity_ma12',
    'quantity_std6'
]
target = 'order_quantity'
X = modeling_data[features]
y = modeling_data[target]

models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(random_state=42, n_jobs=-1),
    'XGBoost': XGBRegressor(random_state=42, n_jobs=-1)
}
param_grids = {
    'Linear Regression': {},
    'Random Forest': {'n_estimators': [100, 200], 'max_depth': [10, 20]},
    'XGBoost': {'n_estimators': [100, 200], 'learning_rate': [0.05, 0.1], 'max_depth': [3, 5]}
}

print("Validação cruzada")
n_splits = 5
tscv = TimeSeriesSplit(n_splits=n_splits)
results = {name: [] for name in models.keys()}
results['Baseline (Média)'] = []
last_fold_data = {}

for fold, (train_index, test_index) in enumerate(tscv.split(X)):
    print(f"fold {fold + 1}/{n_splits}")
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    train_dates = modeling_data['order_date'].iloc[train_index]
    test_dates = modeling_data['order_date'].iloc[test_index]
    print(f"Treino: {len(X_train)} amostras (de {train_dates.min().date()} a {train_dates.max().date()})")
    print(f"Teste:  {len(X_test)} amostras (de {test_dates.min().date()} a {test_dates.max().date()})")

    baseline_pred = np.full(len(y_test), y_train.mean())
    baseline_mape = mean_absolute_percentage_error(y_test, baseline_pred) * 100
    results['Baseline (Média)'].append(baseline_mape)

    print(f"- Baseline (Média) MAPE: {baseline_mape:.2f}%")

    for name, model in models.items():
        if not param_grids[name]:
            model.fit(X_train, y_train)
            best_model = model
        else:
            grid_search = GridSearchCV(estimator=model, param_grid=param_grids[name], cv=3,
                                       scoring='neg_mean_absolute_percentage_error', n_jobs=-1)
            grid_search.fit(X_train, y_train)
            best_model = grid_search.best_estimator_

        pred = best_model.predict(X_test)
        mape = mean_absolute_percentage_error(y_test, pred) * 100
        results[name].append(mape)
        print(f"- {name} MAPE: {mape:.2f}%")
    
        if fold == n_splits - 1:
            last_fold_data['y_test'] = y_test
            last_fold_data['predictions'] = {}
            last_fold_data['predictions']['Baseline (Média)'] = baseline_pred
            for name, model in models.items():
                if not param_grids[name]:
                    model.fit(X_train, y_train)
                    last_fold_data['predictions'][name] = model.predict(X_test)
                else:
                    grid_search = GridSearchCV(estimator=model, param_grid=param_grids[name], cv=3, scoring='neg_mean_absolute_percentage_error', n_jobs=-1)
                    grid_search.fit(X_train, y_train)
                    last_fold_data['predictions'][name] = grid_search.best_estimator_.predict(X_test)


print("Resultados")
summary = []
for name, mapes in results.items():
    mean_mape = np.mean(mapes)
    std_mape = np.std(mapes)
    summary.append({'Modelo': name, 'MAPE medio': mean_mape, 'Desvio padrão': std_mape})

results_df = pd.DataFrame(summary).sort_values(by='MAPE medio').reset_index(drop=True)
print(results_df.to_string(index=False, formatters={'MAPE medio':'{:.2f}%'.format, 'Desvio Padrão':'{:.2f}'.format}))

best_model_name = results_df[results_df['Modelo'] != 'Baseline (Média)'].iloc[0]['Modelo']

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

ax1 = axes[0]
bars = ax1.bar(results_df['Modelo'], results_df['MAPE medio'], alpha=0.9)
ax1.set_title('Performance dos modelos (MAPE Médio)', fontweight='bold', fontsize=14)
ax1.set_ylabel('MAPE médio (%) - erro percentual', fontsize=12)
ax1.set_xticklabels(results_df['Modelo'], rotation=20, ha='right')
ax1.grid(axis='y', linestyle='--', alpha=0.7)

for bar in bars:
    yval = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2.0, yval + 0.5, f'{yval:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax2 = axes[1]
y_test_viz = last_fold_data['y_test']
best_pred_viz = last_fold_data['predictions'][best_model_name]

ax2.scatter(y_test_viz, best_pred_viz, alpha=0.6, s=40, edgecolor='k', c='gold')

ax2.plot([y_test_viz.min(), y_test_viz.max()], [y_test_viz.min(), y_test_viz.max()], 'r--', lw=2, label='Linha ideal')

m, b = np.polyfit(y_test_viz, best_pred_viz, 1)
x_fit = np.array([y_test_viz.min(), y_test_viz.max()]) 
ax2.plot(x_fit, m*x_fit + b, 'b-', lw=2, label='Linha de Regressão') 

ax2.set_title(f'Predições vs. Real ({best_model_name}) - Último fold', fontweight='bold', fontsize=14)
ax2.set_xlabel('Valores reais', fontsize=12)
ax2.set_ylabel('Valores previstos', fontsize=12)
ax2.legend()
ax2.grid(True, linestyle='--', alpha=0.7)

r2_best = r2_score(y_test_viz, best_pred_viz)
ax2.text(0.05, 0.85, f'R² = {r2_best:.3f}', transform=ax2.transAxes, fontsize=12,
         verticalalignment='top', bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8))

fig.suptitle('Viabilidade dos modelos de regressão', fontsize=18, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

best_mape = results_df[results_df['Modelo'] == best_model_name]['MAPE medio'].values[0]
baseline_mape = results_df[results_df['Modelo'] == 'Baseline (Média)']['MAPE medio'].values[0]
improvement = (baseline_mape - best_mape) / baseline_mape * 100

recommendation = (f"O modelo '{best_model_name}' é o mais recomendado, "
                  f"apresentando um MAPE Médio de {best_mape:.1f}% na validação cruzada. "
                  f"Uma melhoria de {improvement:.1f}% em relação ao Baseline.")

print("Recomendação:")
print(recommendation)